In [91]:
import pandas as pd
import numpy as np

# =========================
# CONFIG
# =========================
INPUT_PATH = "Zınarp.csv"
OUTPUT_PATH = "moving_avg_fixed_repeats_Zınarp.csv"

WINDOW_SIZE = 5          # moving average pencere boyutu (3/5/7 deneyebilirsiniz)
CENTERED = True          # True: pencereyi merkeze alır (daha dengeli doldurma)
META_CANDIDATES = {"elements", "element", "id", "time", "timestamp"}  # varsa koruZınarp

# =========================
# Robust CSV loader
# =========================
def read_csv_robust(path: str) -> pd.DataFrame:
    for sep in [";", ",", "\t"]:
        try:
            df_try = pd.read_csv(path, sep=sep)
            if df_try.shape[1] > 1:
                print(f"Loaded with sep='{sep}' -> shape={df_try.shape}")
                return df_try
        except Exception:
            continue
    df = pd.read_csv(path)
    print(f"Loaded with default separator -> shape={df.shape}")
    return df

# =========================
# Rolling mean filler
# =========================
def fill_na_with_rolling_mean(s: pd.Series, w: int, centered: bool) -> pd.Series:
    roll = s.rolling(window=w, min_periods=1, center=centered).mean()
    return s.fillna(roll)

# =========================
# MAIN
# =========================
df_raw = read_csv_robust(INPUT_PATH)

# Meta sütunları ayır (varsa)
meta_cols = [c for c in df_raw.columns if str(c).strip().lower() in META_CANDIDATES]
df_meta = df_raw[meta_cols].copy() if meta_cols else None

# Sadece EEG sayısal sütunlar
df = df_raw.drop(columns=meta_cols, errors="ignore").copy()
df = df.apply(pd.to_numeric, errors="coerce")

# 1) Ardışık tekrarları NaN yap
#    (Bir sensör aynı değerde "takılı" kaldığında bu yaklaşımla temizlenir)
for col in df.columns:
    dup_mask = df[col].eq(df[col].shift(1))
    df.loc[dup_mask, col] = np.nan

# 2) NaN'ları moving average ile doldur
df = df.apply(lambda s: fill_na_with_rolling_mean(s, WINDOW_SIZE, CENTERED))

# 3) KenarlZınarp / uzun boşluklZınarp kalabilecek NaN’lar için güvenlik:
#    önce interpolasyon, sonra ffill/bfill
df = df.interpolate(method="linear", axis=0, limit_direction="both")
df = df.replace([np.inf, -np.inf], np.nan)
df = df.ffill().bfill()

# 4) Çıkışı kaydet (meta sütunları varsa başa ekle)
df_out = pd.concat([df_meta, df], axis=1) if df_meta is not None else df
df_out.to_csv(OUTPUT_PATH, index=False)

# 5) Kısa kontrol
nan_count = int(df_out.isna().sum().sum())
print("Saved:", OUTPUT_PATH)
print("Remaining NaN count:", nan_count)
print(df_out.head())


Loaded with sep=';' -> shape=(83, 21)
Saved: moving_avg_fixed_repeats_Zınarp.csv
Remaining NaN count: 0
   Elements  Delta_TP9  Delta_AF7  Delta_AF8  Delta_TP10  Theta_TP9  \
0  Baseline   0.579579   0.679387   0.583039    1.326727   0.129464   
1  Baseline   0.496747   0.796528   0.535879    1.013703  -0.053335   
2  Baseline   0.321546   0.815764   0.571825    0.895464  -0.141077   
3  Baseline   0.386079   0.809546   0.665755    0.700680   0.777399   
4  Baseline   0.361620   0.921921   0.775985    0.658986   0.601676   

   Theta_AF7  Theta_AF8  Theta_TP10  Alpha_TP9  ...  Alpha_AF8  Alpha_TP10  \
0   0.610133   0.689365    0.877260   0.757350  ...   0.878085    1.262580   
1   0.639121   0.756173    0.754453  -0.066693  ...   0.867445    1.013500   
2   0.654235   0.762299    0.693059  -0.240366  ...   0.814469    0.960015   
3   0.542233   0.820436    0.631646   0.844448  ...   0.788037    0.764420   
4   0.512834   0.694054    0.570270   0.888821  ...   0.820334    0.853045   



import pandas as pd
import numpy as np

# =========================
# LOAD
# =========================
file_path = "moving_avg_fixed_repeats_Zınarp.csv"
df = pd.read_csv(file_path)

EPS = 1e-12

# -------------------------
# Identify EEG band columns
# -------------------------
bands = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
band_cols = [c for c in df.columns if any(c.startswith(b) for b in bands)]

# Convert ONLY band columns to numeric (avoid warnings)
df[band_cols] = df[band_cols].apply(pd.to_numeric, errors="coerce")

# -------------------------
# Split baseline vs conditions (use .copy() to avoid SettingWithCopyWarning)
# -------------------------
baseline_df = df.loc[df["Elements"].astype(str).str.lower().eq("baseline"), band_cols].copy()
cond_df = df.loc[df["Elements"].astype(str).str.lower().isin(["Empty", "WithPeople"])].copy()

if baseline_df.empty:
    raise ValueError("No baseline rows found. Check 'Elements' values (e.g., 'Baseline').")

if cond_df.empty:
    raise ValueError("No condition rows found for Empty/WithPeople. Check 'Elements' values.")

# -------------------------
# Baseline mean (global)
# If you later have Subject/Trial, we can do groupby-baseline per subject/trial.
# -------------------------
baseline_mean = baseline_df.mean(axis=0, skipna=True)

# -------------------------
# Baseline-normalized percent change for ALL band columns (vectorized)
# BLpct = (X - BL) / (|BL| + eps)
# -------------------------
cond_df_band = cond_df[band_cols].copy()
cond_df_blpct = (cond_df_band - baseline_mean) / (baseline_mean.abs() + EPS)

# attach normalized columns with suffix
for c in band_cols:
    cond_df[f"{c}_BLpct"] = cond_df_blpct[c]

print("Baseline-normalized band power computed (BLpct).")
print(cond_df[[f"{c}_BLpct" for c in band_cols]].head())

# =========================
# AROUSAL / VALENCE (baseline-rZınarprenced)
# Using AF7/AF8 frontal channels
# =========================
required = ["Alpha_AF7_BLpct", "Alpha_AF8_BLpct", "Beta_AF7_BLpct", "Beta_AF8_BLpct"]
missing = [c for c in required if c not in cond_df.columns]
if missing:
    raise KeyError(f"Missing required columns for A/V: {missing}")

Alpha_L = cond_df["Alpha_AF7_BLpct"]
Alpha_R = cond_df["Alpha_AF8_BLpct"]
Beta_L  = cond_df["Beta_AF7_BLpct"]
Beta_R  = cond_df["Beta_AF8_BLpct"]

# Valence: frontal alpha asymmetry-like difference (your earlier definition)
cond_df["Valence_BL"] = Alpha_L - Alpha_R

# Arousal: (Beta_L + Beta_R) / (Alpha_L + Alpha_R)
cond_df["Arousal_BL"] = (Beta_L + Beta_R) / (Alpha_L + Alpha_R + EPS)

# =========================
# SAVE
# =========================
output_path = "arousal_valence_baseline_rZınarprenced_Zınarp.csv"
cond_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print(cond_df[["Elements", "Arousal_BL", "Valence_BL"]].head())


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# =========================
# LOAD
# =========================
file_path = "arousal_valence_baseline_rZınarprenced_Zınarp.csv"
df = pd.read_csv(file_path)

# =========================
# FILTER: ONLY EMPTY CONDITION
# =========================
empty_df = df.loc[df["Elements"] == "WithPeople"].copy()

if empty_df.empty:
    raise ValueError("No 'WithPeople' condition rows found!")

# =========================
# 1) DESCRIPTIVE STATISTICS
# =========================
desc_df = (
    empty_df[["Arousal_BL", "Valence_BL"]]
    .agg(["mean", "std", "median", "count"])
    .transpose()
    .reset_index()
    .rename(columns={"index": "Metric"})
)

desc_output = "WithPeople_descriptive_vs_baselineZınarp.csv"
desc_df.to_csv(desc_output, index=False)

# =========================
# 2) ONE-SAMPLE T-TEST
# H0: mean = 0 (baseline)
# =========================
ttest_results = []

for metric in ["Arousal_BL", "Valence_BL"]:
    values = empty_df[metric].dropna()

    if len(values) < 2:
        continue

    t, p = stats.ttest_1samp(values, 0.0)

    ttest_results.append({
        "Condition": "WithPeople",
        "Metric": metric,
        "Mean": values.mean(),
        "Std": values.std(),
        "t_stat": t,
        "p_value": p,
        "Direction_vs_Baseline": "Increase" if values.mean() > 0 else "Decrease"
    })

ttest_df = pd.DataFrame(ttest_results)

ttest_output = "WithPeople_ttest_vs_baselinZınarp.csv"
ttest_df.to_csv(ttest_output, index=False)

# =========================
# DONE
# =========================
print("EMPTY condition analysis saved:\n")
print("1)", desc_output)
print("2)", ttest_output)

print("\nPreview – Descriptive statistics:")
print(desc_df)

print("\nPreview – One-sample t-test vs baseline:")
print(ttest_df)
